# RedZone Teacher 32B: causal per-play commentary (EXP-1402, Data planner E; self-contained)

The 32B Teacher writes one line of commentary per live play: importance 0-10, lock_in, the most likely next thing,
and confidence. It sees **only the brief at that play** (what the live app knows), never what happened next, so the
answer is causal and the world model may read it as context. TRAIN nights only; valid, test and locked never asked.
The answers land as teacher-v1 tokens (`scripts/synth_build.py teacher`), tagged synthetic and
**personal-use, not for commercial release** (plan B, Nick 11:20 PM Nov 4).

No Drive, no secrets, nothing to upload: code comes from two public commits (kaggle-inputs for the moment builder and
the Teacher runner, the Data E branch for the queue and tokens).

Runtime: **A100 GPU**, then Runtime > **Run all**. ~3.3 h, **~43 CU** (build 1 h, Teacher 2 h, tokens + grade a few
min). If the runtime restarts but keeps its disk, Run all again: moments and answers are kept. The last cell
prints the grade and the CU spent, and downloads commentary_out.zip (hand it to Data planner E or lane 1).

In [ ]:
KPIN = "eac55da9b50f346a39dc8df393b7dc568fcd6bc7"   # kaggle-inputs: code pack (moment builder, run_teacher.py)
EPIN = "d4c0adbde553f48c79e9f57bd2dfe6a7f339b1be"   # claude/data-e-synthetic: commentary_queue.py, synth_build.py
REPO = "BouncySlime1215/redzone-everything"
DATA_PLAN = "B"                 # Nick 11:20 PM Nov 4: personal use only
LEAGUES = "nfl cfb nba cbb nhl mlb"
BUILD_H = 1.0                   # moment downloads (CPU) before the Teacher starts
TEACHER_H = 2.0                 # Teacher budget; your-team, crunch and late plays go first, leagues round-robin
PER_LEAGUE = 0                  # 0 = every train play (the budget decides how many get answered)
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}

In [ ]:
import gzip, json, os, subprocess, sys, tarfile, time, urllib.request, zipfile
T_START = time.time()
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)
RUN = "/content/rzcomm"
os.makedirs(RUN, exist_ok=True)

def fetch(ref, dst, keep=lambda p: True):
    if os.path.exists(f"{dst}/.done"):
        return
    os.makedirs(dst, exist_ok=True)
    for i in range(4):
        try:
            urllib.request.urlretrieve(f"https://codeload.github.com/{REPO}/tar.gz/{ref}", f"{RUN}/x.tar.gz")
            break
        except Exception as e:
            print(" retry", type(e).__name__, e); time.sleep(20 * (i + 1))
    else:
        raise SystemExit("could not download the code; Run all again in a minute")
    with tarfile.open(f"{RUN}/x.tar.gz") as t:
        for m in t.getmembers():
            parts = m.name.split("/", 1)
            if len(parts) == 2 and parts[1] and ".." not in parts[1].split("/") and keep(parts[1]):
                m.name = parts[1]
                t.extract(m, dst, filter="data")
    os.remove(f"{RUN}/x.tar.gz")
    open(f"{dst}/.done", "w").close()

PACK = f"{RUN}/kaggle-inputs"
fetch(KPIN, PACK, keep=lambda p: p.endswith("rz_code.tar.gz") or p.startswith("data/reasoner") or p == "manifest.json")
CODE = f"{RUN}/rzcode"
if not os.path.exists(f"{CODE}/code/agent/kaggle/run_teacher.py"):
    with tarfile.open(f"{PACK}/rz_code.tar.gz") as t:
        t.extractall(CODE, filter="data")
E = f"{RUN}/data_e"
fetch(EPIN, E, keep=lambda p: p.startswith(("redzone/", "scripts/synth_build.py", "agent/kaggle/commentary_queue.py",
                                             "agent/__init__.py", "agent/kaggle/__init__.py")))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "httpx", "pyyaml", "lightgbm"], check=False)
print("code ready")

In [ ]:
env = {**os.environ, "RZ_DATA_PLAN": DATA_PLAN, "RZ_CODE_PACK": f"kaggle-inputs@{KPIN}", "PYTHONUNBUFFERED": "1",
       "PYTHONPATH": f"{CODE}/code"}
OUT = f"{RUN}/out"; MOM = f"{OUT}/moments"
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, f"{CODE}/code")
from agent.kaggle.teacher_only import gate
gate(DATA_PLAN, "espn")          # ESPN moments are plan B only
if not os.path.exists(f"{MOM}/DONE"):
    subprocess.run([sys.executable, "-u", "agent/kaggle/build_moments_kaggle.py", "--out", MOM, "--leagues",
                    *LEAGUES.split(), "--budget-h", str(BUILD_H)], cwd=f"{CODE}/code", env=env, check=True)
q = subprocess.run([sys.executable, f"{E}/agent/kaggle/commentary_queue.py", "--moments", MOM, "--out",
                    f"{OUT}/requests.jsonl", "--leagues", *LEAGUES.split(),
                    *(["--per-league", str(PER_LEAGUE)] if PER_LEAGUE else [])],
                   env={**env, "PYTHONPATH": E}, capture_output=True, text=True)
print(q.stdout, q.stderr[-2000:])

In [ ]:
log_path = f"{RUN}/teacher_{time.strftime('%Y%m%d-%H%M')}.log"
with open(log_path, "w") as log:
    p = subprocess.Popen([sys.executable, "-u", "agent/kaggle/run_teacher.py", "--requests", f"{OUT}/requests.jsonl",
                          "--out", OUT, "--budget-h", str(TEACHER_H)], cwd=f"{CODE}/code", env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); log.write(line)
    print("teacher exit code:", p.wait())

In [ ]:
# tokens (teacher-v1, mode 2 = causal) + the cap check
subprocess.run([sys.executable, f"{E}/scripts/synth_build.py", "teacher", "--answers", f"{OUT}/teacher_out.jsonl",
                "--out", f"{OUT}/SYN"], env={**env, "PYTHONPATH": E}, check=False)
# grade (EXP-1402): valid JSON, and importance vs what REALLY happened next on these train plays (rule 3: real outcomes)
real = {}
for f in os.listdir(f"{MOM}/rows"):
    if f.endswith("_train.jsonl.gz"):
        with gzip.open(f"{MOM}/rows/{f}", "rt") as fh:
            for line in fh:
                r = json.loads(line)
                if r.get("split") == "train":
                    real[r["id"]] = (bool(r.get("big_next")), r.get("league"))
sys.path.insert(0, E)
from redzone.noggin.synth import _json
per = {}
for line in open(f"{OUT}/teacher_out.jsonl"):
    a = json.loads(line); rid = a["id"].split("|")[0]
    big, lg = real.get(rid, (None, rid.split(":")[0]))
    j = _json(a.get("text") or "")
    e = per.setdefault(lg, {"answered": 0, "valid": 0, "pairs": []})
    e["answered"] += 1
    if j is not None and isinstance(j.get("importance"), (int, float)):
        e["valid"] += 1
        if big is not None:
            e["pairs"].append((float(j["importance"]), big))

def auc(pairs):
    pos = [s for s, y in pairs if y]; neg = [s for s, y in pairs if not y]
    if not pos or not neg:
        return None
    import numpy as np
    s = np.array([x for x, _ in pairs]); y = np.array([b for _, b in pairs])
    from scipy.stats import rankdata
    r = rankdata(s)
    return round(float((r[y].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))), 4)

grade = {"exp": "EXP-1402", "license": "personal-use, not for commercial release", "data_plan": DATA_PLAN,
         "kpin": KPIN, "epin": EPIN, "split": "train nights only (build_moments split plan); locked never downloaded",
         "per_league": {lg: {"answered": e["answered"], "valid_json_pct": round(100 * e["valid"] / max(1, e["answered"]), 1),
                             "importance_auc_vs_real_big_next": auc(e["pairs"]), "n_graded": len(e["pairs"])}
                        for lg, e in sorted(per.items())}}
rep = json.load(open(f"{OUT}/teacher_report.json")) if os.path.exists(f"{OUT}/teacher_report.json") else {}
wall_h = (time.time() - T_START) / 3600
grade.update(teacher=rep, colab={"gpu": GPU, "wall_h": round(wall_h, 2), "cu_spent": round(wall_h * RATE, 1) if RATE else None})
json.dump(grade, open(f"{OUT}/grades.json", "w"), indent=1)
print(json.dumps(grade, indent=1))

In [ ]:
zp = f"{RUN}/commentary_out.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("teacher_out.jsonl", "grades.json", "teacher_report.json"):
        if os.path.exists(f"{OUT}/{f}"):
            z.write(f"{OUT}/{f}", f)
    for root, _, files in os.walk(f"{OUT}/SYN"):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print("zip:", zp, round(os.path.getsize(zp) / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zp)
except Exception as e:
    print("commentary_out.zip is at", zp, f"({type(e).__name__})")